In [ ]:
import itertools
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
import re
from spopt.region import Skater, WardSpatial
from pathlib import Path

In [ ]:
shapes = ['square', 'pent', 'hex']
sizes = [100, 400]
c_runs = range(10)                            # Corruption runs 0-9
z_runs = range(10)                            # Zonations 0-9
n_variables = 5                               # Number of attributes per zonation
methods = ["random","local","borders"]
scaler = StandardScaler()
GRAPH_NAME_RE = re.compile(r"g_missing(\d+)_run(\d+)\.parquet$")

In [ ]:
def parse_missing_edges(graph_path):
    m = GRAPH_NAME_RE.search(str(graph_path))
    if not m:
        raise ValueError(f"could not parse missing-edge count from: {graph_path}")
    return int(m.group(1))
 
 
def get_available_intensities(shape, size, method, c_run, subfolder):
    directory = Path(f"graphs/{shape}/size_{size}/{subfolder}/{method}")
    found = []
    for f in directory.glob(f"g_missing*_run{c_run}.parquet"):
        m = GRAPH_NAME_RE.match(f.name)
        if m:
            found.append(int(m.group(1)))
    return sorted(found)
 
 
def select_sample_graphs(available_intensities, n_sample=20):
    available_intensities = sorted(available_intensities)
    max_idx = len(available_intensities) - 1
    idxs = np.linspace(0, max_idx, n_sample, dtype=int)
    idxs = idxs[1:]
    idxs = sorted(set(idxs))
    return [available_intensities[i] for i in idxs]

In [ ]:
def _process_regionalization(shape, size, intensity, c_run, z_run, method,
                              gdf_geometry, gdf_attributes, n_variables):
    tag = f"[{shape} {size} {method} c{c_run} z{z_run} i{intensity}]"
 
    n = len(gdf_geometry)
    cluster_variables = [f"zone_run{z_run}_run{v}" for v in range(n_variables)]
    var_scaled = StandardScaler().fit_transform(gdf_attributes[cluster_variables].values)
 
    ward_labels   = np.full(n, -1, dtype=np.int8)
    skater_labels = np.full(n, -1, dtype=np.int8)
 
    graph_path = f"graphs/{shape}/size_{size}/one_component/{method}/g_missing{intensity}_run{c_run}.parquet"
    missing_edges = parse_missing_edges(graph_path)
 
    try:
        w = libpysal.graph.read_parquet(graph_path).to_W()
    except FileNotFoundError:
        print(f"{tag} MISSING graph file, skipping", flush=True)
        return _build_result_df(gdf_geometry, method, missing_edges, c_run, z_run, ward_labels, skater_labels)
 
    local_gdf = gpd.GeoDataFrame(geometry=gdf_geometry)
    temp_cols = [f"v_{i}" for i in range(n_variables)]
    local_gdf[temp_cols] = var_scaled
 
    try:
        model_ward = WardSpatial(gdf=local_gdf, w=w, attrs_name=temp_cols, n_clusters=5)
        model_ward.solve()
        ward_labels = model_ward.labels_.astype(np.int8)
    except Exception as e:
        print(f"{tag} ward failed: {e}", flush=True)
 
    try:
        model_skater = Skater(gdf=local_gdf, w=w, attrs_name=temp_cols, n_clusters=5, floor=1, islands="ignore")
        model_skater.solve()
        skater_labels = model_skater.labels_.astype(np.int8)
    except Exception as e:
        print(f"{tag} skater failed: {e}", flush=True)

 
    return _build_result_df(gdf_geometry, method, missing_edges, c_run, z_run, ward_labels, skater_labels)
 
 
def _build_result_df(gdf_geometry, method, missing_edges, c_run, z_run, ward_labels, skater_labels):
    n = len(gdf_geometry)
    return pd.DataFrame({
        "obs_id"       : gdf_geometry.index.values,
        "method"       : pd.Categorical([method] * n, categories=methods),
        "missing_edges": np.full(n, missing_edges, dtype=np.int32),  # int16 caps at 32,767 - too small for
                                                                       # large grids (square400 rook graph
                                                                       # has ~319k edges, queen ~637k+)
        "c_run"        : np.full(n, c_run,      dtype=np.int8),
        "z_run"        : np.full(n, z_run,      dtype=np.int8),
        "ward_label"   : ward_labels,
        "skater_label" : skater_labels,
    })
 
 

In [ ]:
N_SAMPLE = 20
N_JOBS = -1
 
for shape, size in itertools.product(shapes, sizes):
    gdf = gpd.read_parquet(f'data/regionalization/gdf_{shape}_{size}.parquet')
    gdf_geometry = gdf.geometry.copy()
    all_needed_columns = [f"zone_run{z}_run{v}" for z in z_runs for v in range(n_variables)]
    gdf_attributes = gdf[all_needed_columns].copy()
 
    task_combinations = []
    for method in methods:
        for c_run in c_runs:
            available = get_available_intensities(shape, size, method, c_run, subfolder="one_component")
            sampled = select_sample_graphs(available, n_sample=N_SAMPLE)
            for intensity in sampled:
                for z_run in z_runs:
                    task_combinations.append((intensity, c_run, z_run, method)) 
    results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(_process_regionalization)(
            shape, size, intensity, c_run, z_run, method, gdf_geometry, gdf_attributes, n_variables
        )
        for intensity, c_run, z_run, method in task_combinations
    )
 
    res_df = pd.concat(results, ignore_index=True)
    output_path = f"results/regionalization/synthetic/reg_{shape}_{size}.parquet"
    res_df.to_parquet(output_path, index=False)
    print(f"Saved: {output_path}  |  shape: {res_df.shape}  |  tasks: {len(task_combinations)}\n", flush=True)

In [ ]:
result = pd.read_parquet(f'results/regionalization/synthetic/reg_hex_400.parquet')
result

In [ ]:
result.plot(column = "zonation9_agg_cintensity18_crun_9")

In [ ]:
result.plot(column = "zonation9_skater_cintensity18_crun_9")